# Search Videos

Find specific moments, topics, or content across your video collection with natural language.
Use this for locating content in a large collection, matching moments to a description, or building search features in your app.

In [ ]:
import json
import os

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(result: dict) -> str | dict:
    """Extract text content from a Jockey API response."""
    for output in result["output"]:
        if output["type"] == "message":
            for content in output["content"]:
                return content["text"]
    return ""

## Search Schema

Define a JSON schema for structured search results. Each result includes a video reference,
timestamp, description, and relevance score. The schema also captures total result count
and how Jockey interpreted your query.

In [ ]:
SEARCH_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "video_reference": {"type": "string"},
                    "timestamp": {"type": "string"},
                    "description": {"type": "string"},
                    "relevance": {"type": "string"},
                },
            },
        },
        "total_results": {"type": "integer"},
        "query_interpretation": {"type": "string"},
    },
}

## Run a Search

Send a natural-language query to find matching moments across all videos in your knowledge store.
The structured response makes it easy to iterate over results programmatically.

In [ ]:
SEARCH_QUERY = "Find all moments where someone is presenting to an audience"

response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {"type": "message", "role": "user", "content": SEARCH_QUERY}
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "search_results", "schema": SEARCH_SCHEMA}},
    },
)

result = response.json()
search = json.loads(parse_response(result))

print(f"Found {search['total_results']} results")
print(f"Interpreted as: {search['query_interpretation']}\n")

for r in search["results"]:
    print(f"  [{r['timestamp']}] {r['video_reference']}")
    print(f"    {r['description']}")
    print(f"    Relevance: {r['relevance']}\n")

## Example Queries

| Query | What It Finds |
|-------|---------------|
| "someone laughing" | Moments with laughter |
| "product being held up to camera" | Product showcase moments |
| "outdoor scenes with water" | Nature/water visuals |
| "heated discussion" | Tense conversational moments |
| "text on screen" | Moments with overlaid text or titles |

## Variations

- **Narrow by context:** Add instructions like "Only search the first 2 minutes of each video"
- **Ranked results:** "Find and rank the top 5 most visually striking moments"
- **Multi-turn refinement:** Search, then follow up with "Show me more like the third result"

## Next Steps

- **[Get Corpus Overview](get_corpus_overview.ipynb)** -- understand what's in your collection first
- **[Extract Entities](extract_entities.ipynb)** -- list all people, places, objects, and concepts
- **[Find Organization Axes](find_organization_axes.ipynb)** -- discover the best categorization strategies
- **[Enrich Content](enrich_content.ipynb)** -- get deeper, domain-specific analysis

See also:
- [Querying Guide](../../docs/guides/querying.md) -- fundamentals of the Responses API